# Lab 8: Python–JavaScript Comparison
Alex Bebb — PHYS 311

## Purpose
I compare the JavaScript projectile simulation with my existing Lab 7
Python engine. Both versions use the same initial conditions, gravity,
time step, and number of steps.

Each simulation advances a 1 kg projectile launched at 20 m/s and
50 degrees through 1,200 steps of 1/120 second. There is no ground
collision, so the projectile continues below its initial height.
I compare the final position, velocity, and total mechanical energy
for Euler-Cromer and Velocity Verlet.

In [1]:
import json
import math

from lab07_engine.body import Body
from lab07_engine.forces import Gravity
from lab07_engine.integrators import EulerCromer, VelocityVerlet
from lab07_engine.world import World

## Run the matching Python simulations

The function creates a fresh world and projectile for each method.
It applies uniform gravity and advances the existing engine exactly
1,200 times. It returns the final position, velocity, and energy
without rounding them for display.

In [2]:
def run_python_projectile(integrator, steps=1200, dt=1 / 120):
    speed = 20.0
    angle = 50 * math.pi / 180

    world = World(integrator=integrator)

    body = Body(
        mass=1.0,
        position=(0.0, 0.0),
        velocity=(
            speed * math.cos(angle),
            speed * math.sin(angle)
        ),
        radius=0.3
    )

    world.add_body(body)
    world.add_force(Gravity(g=9.81))

    # Advance exactly the same number of steps as JavaScript.
    for step in range(steps):
        world.step(dt)

    return {
        "steps": steps,
        "dt": dt,
        "t": steps * dt,
        "x": float(body.position[0]),
        "y": float(body.position[1]),
        "vx": float(body.velocity[0]),
        "vy": float(body.velocity[1]),
        "E": float(world.total_energy())
    }


python_results = {
    "Euler-Cromer": run_python_projectile(EulerCromer()),
    "Velocity Verlet": run_python_projectile(VelocityVerlet())
}

print(json.dumps(python_results, indent=2))

{
  "Euler-Cromer": {
    "steps": 1200,
    "dt": 0.008333333333333333,
    "t": 10.0,
    "x": 128.5575219373096,
    "y": -337.69986137620293,
    "vx": 12.855752193730787,
    "vy": -82.77911113762003,
    "E": 195.99016249998067
  },
  "Velocity Verlet": {
    "steps": 1200,
    "dt": 0.008333333333333333,
    "t": 10.0,
    "x": 128.5575219373096,
    "y": -337.2911113762022,
    "vx": 12.855752193730787,
    "vy": -82.77911113761996,
    "E": 199.9999999999818
  }
}


## Compare Python and JavaScript

I compare the final position, velocity, and total energy from both
implementations. The JavaScript values below were copied from the
browser console. For each quantity, I calculate the absolute difference
between the Python and JavaScript results.

In [3]:
from IPython.display import Markdown, display

# Values from the JavaScript browser console.
javascript_results = {
    "Euler-Cromer": {
        "x": 128.5575219373096,
        "y": -337.69986137620293,
        "vx": 12.855752193730787,
        "vy": -82.77911113762003,
        "E": 195.99016249998067
    },
    "Velocity Verlet": {
        "x": 128.5575219373096,
        "y": -337.2911113762022,
        "vx": 12.855752193730787,
        "vy": -82.77911113761996,
        "E": 199.9999999999818
    }
}

units = {
    "x": "m",
    "y": "m",
    "vx": "m/s",
    "vy": "m/s",
    "E": "J"
}

for method, js_values in javascript_results.items():
    rows = [
        f"### {method}",
        "",
        "| Quantity | Python | JavaScript | Absolute difference |",
        "|---|---:|---:|---:|"
    ]

    for quantity, unit in units.items():
        python_value = python_results[method][quantity]
        javascript_value = js_values[quantity]
        difference = abs(python_value - javascript_value)

        rows.append(
            f"| {quantity} ({unit}) "
            f"| {python_value:.17g} "
            f"| {javascript_value:.17g} "
            f"| {difference:.3e} |"
        )

    display(Markdown("\n".join(rows)))

### Euler-Cromer

| Quantity | Python | JavaScript | Absolute difference |
|---|---:|---:|---:|
| x (m) | 128.55752193730959 | 128.55752193730959 | 0.000e+00 |
| y (m) | -337.69986137620293 | -337.69986137620293 | 0.000e+00 |
| vx (m/s) | 12.855752193730787 | 12.855752193730787 | 0.000e+00 |
| vy (m/s) | -82.779111137620035 | -82.779111137620035 | 0.000e+00 |
| E (J) | 195.99016249998067 | 195.99016249998067 | 0.000e+00 |

### Velocity Verlet

| Quantity | Python | JavaScript | Absolute difference |
|---|---:|---:|---:|
| x (m) | 128.55752193730959 | 128.55752193730959 | 0.000e+00 |
| y (m) | -337.2911113762022 | -337.2911113762022 | 0.000e+00 |
| vx (m/s) | 12.855752193730787 | 12.855752193730787 | 0.000e+00 |
| vy (m/s) | -82.779111137619964 | -82.779111137619964 | 0.000e+00 |
| E (J) | 199.99999999998181 | 199.99999999998181 | 0.000e+00 |

## Interpretation

The Python and JavaScript results agree for both methods, with zero
difference in the compared values for this test. This supports that
the JavaScript port reproduces the Lab 7 engine's calculations for
a projectile under uniform gravity.

Agreement between languages does not mean that both methods give
the exact physical answer. Euler-Cromer loses about 4.010 J over
the 10-second simulation, while Velocity Verlet keeps the energy
at approximately 200 J. Both languages reproduce the same numerical
behavior, including Euler-Cromer's error.

This checks one physical situation and set of initial conditions.
Other forces and systems would need their own validation.

## Part C2: Fixed versus variable time steps

All four runs use Euler-Cromer with the same initial conditions.
The values below are rounded readings from the browser HUD.

| Time stepping | Added lag | Final time (s) | Final x (m) | Final energy (J) |
|---|---|---:|---:|---:|
| Fixed | No | 3.117 | 40.067 | 198.750 |
| Fixed | Yes | 3.117 | 40.067 | 198.750 |
| Variable | No | 3.113 | 40.022 | 197.506 |
| Variable | Yes | 3.080 | 39.594 | 192.331 |

The fixed-step runs gave the same final results with and without
added lag because the accumulator divided elapsed time into the
same small physics steps. With variable stepping, slower frames
produced larger physics steps, increasing Euler-Cromer's numerical
energy loss from 2.494 J to 7.669 J in these runs. This energy loss
is numerical error, since the model includes no drag or other
dissipative force. The runs stop at the first below-ground position,
so their final positions are not comparisons at exactly the same
simulation time. These results show why separating physics time
steps from drawing frames makes the simulation more repeatable.

## Part C3: Comparing integrators at a larger time step

Both runs use a fixed time step of 1/20 s with no added lag.

| Method | Final time (s) | Final energy (J) | Energy loss (J) |
|---|---:|---:|---:|
| Euler-Cromer | 3.100 | 192.542 | 7.458 |
| Velocity Verlet | 3.150 | 200.000 | 0.000 at displayed precision |

At this larger time step, Euler-Cromer loses about 7.458 J, while
Velocity Verlet preserves the initial 200 J to the displayed precision.
Constant gravity is a particularly favorable test for Verlet because
its updates reproduce constant-acceleration motion apart from rounding;
a spring oscillator or gravitational orbit would better reveal its
remaining numerical error over many cycles.
The final times differ because each simulation stops at its first
stored position below ground, rather than locating the exact crossing.